<a href="https://colab.research.google.com/github/trialdyk/marchine-learning/blob/main/JS09/JS09_Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum Pembelajaran Mesin - JS09 Approximate Nearest Neighbors (ANN)
## Lab 4: ANNOY vs. FAISS vs. HNSW

Percobaan kali ini kita akan melihat perbedaan ketiga model yang telah kita bahas dan bandingkan hasilnya.

### Langkah 1 - Instal Library

In [1]:
!pip install annoy faiss-cpu hnswlib

### Langkah 2 - Benchmark ANNOY vs. FAISS vs. HNSW

In [2]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# Build 1mio rows with 5D
# ===============================
n_data = 1_000_000 # try 100_000 if you have limited memory
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')
start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10) # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)
start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)
start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)
start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")

=== Annoy ===


Build time: 11.5374116897583 detik
Query time: 0.001112222671508789 detik
Neighbors: [460827, 441229, 86818, 402131, 979963] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.029679298400878906 detik
Query time: 0.00898122787475586 detik
Neighbors: [460827 441229  86818 402131 979963] ...

=== HNSW (hnswlib) ===


Build time: 92.22077918052673 detik
Query time: 0.0009126663208007812 detik
Neighbors: [460827 441229  86818 402131 979963] ...
